In [1]:
!pip install -q spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 10.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import spacy            #import libraries
import pandas as pd
import ast

In [3]:
nlp = spacy.load("en_core_web_sm")    #load spacy model

print("spaCy model loaded successfully!")

spaCy model loaded successfully!


In [16]:
df = pd.read_csv("/content/combined_jobs_skills_taxonomy (1).csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (8785, 14)

Columns:
['Job Id', 'Job Title', 'Company', 'location', 'Job Description', 'Experience', 'Qualifications', 'Salary Range', 'Work Type', 'skills', 'clean_description', 'matched_skill_taxonomy', 'category', 'aliases']


In [17]:
df.head()

,Job Id,Job Title,Company,location,Job Description,Experience,Qualifications,Salary Range,Work Type,skills,clean_description,matched_skill_taxonomy,category,aliases
0,1.089840e+15,Digital Marketing Specialist,Icahn Enterprises,Douglas,Social Media Managers oversee an organizations...,5 to 15 Years,M.Tech,$59K-$99K,Intern,"Social media platforms (e.g., Facebook, Twitte...",social media managers oversee an organizations...,No Match,No Match,NaN
1,3.984540e+14,Web Developer,PNC Financial Services Group,Ashgabat,Frontend Web Developers design and implement u...,2 to 12 Years,BCA,$56K-$116K,Intern,"HTML, CSS, JavaScript Frontend frameworks (e.g...",frontend web developers design and implement u...,JavaScript; Angular; React; HTML; CSS,Programming; Web Development,".css, .html, angular, angularjs, css, css3, ht..."
2,4.816400e+14,Operations Manager,United Services Automobile Assn.,Macao,Quality Control Managers establish and enforce...,0 to 12 Years,PhD,$61K-$104K,Temporary,Quality control processes and methodologies St...,quality control managers establish and enforce...,No Match,No Match,NaN
3,6.881930e+14,Network Engineer,Hess,Porto-Novo,"Wireless Network Engineers design, implement, ...",4 to 11 Years,PhD,$65K-$91K,Full-Time,Wireless network design and architecture Wi-Fi...,wireless network engineers design implement an...,No Match,No Match,NaN
4,1.170580e+14,Event Manager,Cairn Energy,Santiago,A Conference Manager coordinates and manages c...,1 to 12 Years,MBA,$64K-$87K,Intern,Event planning Conference logistics Budget man...,a conference manager coordinates and manages c...,No Match,No Match,NaN


In [18]:
#standard ner
text = """
Python and AWS are commonly used in data engineering.
Microsoft Azure and Google Cloud are cloud platforms.
New York is a location.
"""

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

AWS -> ORG
Microsoft Azure -> ORG
Google Cloud -> PERSON
New York -> GPE


In [19]:
text = """
The candidate should have experience with Python, SQL, AWS,
PostgreSQL, Power BI, Docker and Machine Learning.
"""

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

SQL -> ORG
AWS -> ORG
PostgreSQL -> GPE
Power BI -> ORG
Docker -> ORG
Machine Learning -> PERSON


In [24]:
taxonomy = df[["skills", "category", "aliases"]].dropna(subset=["skills"])

taxonomy = taxonomy.drop_duplicates()

taxonomy.head(30)

,skills,category,aliases
0,"Social media platforms (e.g., Facebook, Twitte...",No Match,NaN
1,"HTML, CSS, JavaScript Frontend frameworks (e.g...",Programming; Web Development,".css, .html, angular, angularjs, css, css3, ht..."
2,Quality control processes and methodologies St...,No Match,NaN
3,Wireless network design and architecture Wi-Fi...,No Match,NaN
4,Event planning Conference logistics Budget man...,No Match,NaN
5,Quality assurance processes Testing methodolog...,No Match,NaN
6,Teaching pedagogy Classroom management Curricu...,No Match,NaN
7,UI design principles and best practices Graphi...,No Match,NaN
8,Interaction design principles User behavior an...,No Match,NaN
9,Wedding planning Vendor coordination Event man...,No Match,NaN


In [25]:
# PART 1 — Standard (pretrained) spaCy NER
# ---------------------------------------------------------------
nlp_standard = spacy.load("en_core_web_sm")

text = "Python Microsoft AWS Google Cloud New York"
doc = nlp_standard(text)

print(f"{'Entity':<20}{'Label':<10}")
print("-" * 30)
for ent in doc.ents:
    print(f"{ent.text:<20}{ent.label_:<10}")

if not doc.ents:
    print("(no entities detected)")


text2 = ("We need a candidate skilled in Python, AWS, PostgreSQL, and Power BI, "
         "based in New York, who previously worked at Microsoft and Google Cloud.")

doc2 = nlp_standard(text2)

print(f"{'Entity':<20}{'Label':<10}")
print("-" * 30)
for ent in doc2.ents:
    print(f"{ent.text:<20}{ent.label_:<10}")

Entity              Label     
------------------------------
Microsoft           ORG       
Cloud New York      PERSON    
Entity              Label     
------------------------------
Python              GPE       
AWS                 ORG       
PostgreSQL          GPE       
Power BI            ORG       
New York            GPE       
Microsoft           ORG       
Google Cloud        PERSON    


In [26]:
# PART 2 — Custom entity labels & taxonomy
# ---------------------------------------------------------------
SKILL_LABELS = {
    # SKILL — programming languages & core skills
    "Python": "SKILL", "Java": "SKILL", "C++": "SKILL", "JavaScript": "SKILL",
    "HTML": "SKILL", "CSS": "SKILL", "SQL": "SKILL", "R": "SKILL",

    # DATABASE
    "MySQL": "DATABASE", "PostgreSQL": "DATABASE", "MongoDB": "DATABASE", "Oracle": "DATABASE",

    # TOOL — analytics / dev tools
    "Power BI": "TOOL", "Tableau": "TOOL", "Excel": "TOOL", "Docker": "TOOL",
    "Kubernetes": "TOOL", "Git": "TOOL",

    # TECHNOLOGY — frameworks / libraries
    "Pandas": "TECHNOLOGY", "NumPy": "TECHNOLOGY", "Scikit-learn": "TECHNOLOGY",
    "TensorFlow": "TECHNOLOGY", "PyTorch": "TECHNOLOGY", "Spark": "TECHNOLOGY",
    "Hadoop": "TECHNOLOGY", "React": "TECHNOLOGY", "Angular": "TECHNOLOGY",
    "NLTK": "TECHNOLOGY", "spaCy": "TECHNOLOGY", "Transformers": "TECHNOLOGY",

    # CLOUD_PLATFORM
    "AWS": "CLOUD_PLATFORM", "Azure": "CLOUD_PLATFORM", "GCP": "CLOUD_PLATFORM",
}

print(f"Loaded {len(SKILL_LABELS)} custom entities across "
      f"{len(set(SKILL_LABELS.values()))} labels: {sorted(set(SKILL_LABELS.values()))}")

Loaded 33 custom entities across 5 labels: ['CLOUD_PLATFORM', 'DATABASE', 'SKILL', 'TECHNOLOGY', 'TOOL']


In [27]:
# PART 3 — Build custom NER pipeline (EntityRuler)
# ---------------------------------------------------------------
nlp_custom = spacy.blank("en")
ruler = nlp_custom.add_pipe("entity_ruler", config={"phrase_matcher_attr": "LOWER"})

patterns = [{"label": label, "pattern": term} for term, label in SKILL_LABELS.items()]
ruler.add_patterns(patterns)

print(f"Custom NER pipeline ready with {len(patterns)} patterns.")
print("Pipeline components:", nlp_custom.pipe_names)

doc3 = nlp_custom(text2)

print(f"{'Entity':<15}{'Custom Label':<18}")
print("-" * 35)
for ent in doc3.ents:
    print(f"{ent.text:<15}{ent.label_:<18}")


Custom NER pipeline ready with 33 patterns.
Pipeline components: ['entity_ruler']
Entity         Custom Label      
-----------------------------------
Python         SKILL             
AWS            CLOUD_PLATFORM    
PostgreSQL     DATABASE          
Power BI       TOOL              


In [28]:
# PART 4 — Combine custom ruler + standard NER in one pipeline
# ---------------------------------------------------------------
nlp_combined = spacy.load("en_core_web_sm")
ruler2 = nlp_combined.add_pipe("entity_ruler", before="ner",
                                config={"phrase_matcher_attr": "LOWER"})
ruler2.add_patterns(patterns)

doc4 = nlp_combined(text2)
print(f"{'Entity':<15}{'Label':<18}")
print("-" * 35)
for ent in doc4.ents:
    print(f"{ent.text:<15}{ent.label_:<18}")

Entity         Label             
-----------------------------------
Python         SKILL             
AWS            CLOUD_PLATFORM    
PostgreSQL     DATABASE          
Power BI       TOOL              
New York       GPE               
Microsoft      ORG               
Google Cloud   PERSON            


In [41]:
# PART 5 — Apply custom NER to a real dataset at scale
# ---------------------------------------------------------------
jobs = pd.read_csv("/content/combined_jobs_skills_taxonomy (1).csv")
print("Dataset shape:", jobs.shape)
jobs[["Job Description", "clean_description"]].head(10)


def extract_entities(doc):
    """Return list of (entity_text, label) tuples for our custom labels."""
    return [(ent.text, ent.label_) for ent in doc.ents]

texts = jobs["clean_description"].fillna("").astype(str).tolist()

all_entities = []
for doc in nlp_custom.pipe(texts, batch_size=200):
    all_entities.append(extract_entities(doc))

jobs["ner_entities"] = all_entities
print("Entities extracted for", sum(1 for e in all_entities if e), "job postings.")


LABELS = ["SKILL", "TOOL", "TECHNOLOGY", "DATABASE", "CLOUD_PLATFORM"]

for label in LABELS:
    jobs[label] = jobs["ner_entities"].apply(
        lambda ents: "; ".join(sorted({t for t, l in ents if l == label})) or "No Match"
    )

jobs[["Job Description"] + LABELS].head(10)


matched = jobs[LABELS].apply(lambda row: any(v != "No Match" for v in row), axis=1)
print(f"Postings with at least one recognized custom entity: {matched.sum()} / {len(jobs)}")

jobs[matched][["Job Description", "skills"] + LABELS].head(5)

Dataset shape: (8785, 14)
Entities extracted for 119 job postings.
Postings with at least one recognized custom entity: 119 / 8785


,Job Description,skills,SKILL,TOOL,TECHNOLOGY,DATABASE,CLOUD_PLATFORM
98,"SQL Database Developers design, implement, and...",SQL (Structured Query Language) Database desig...,sql,No Match,No Match,No Match,No Match
118,Java Backend Developers specialize in building...,Backend development RESTful APIs Database inte...,java,No Match,No Match,No Match,No Match
129,JavaScript Developers write code to create int...,JavaScript programming Frontend development Fr...,javascript,No Match,No Match,No Match,No Match
154,JavaScript Developers write code to create int...,JavaScript programming Frontend development Fr...,javascript,No Match,No Match,No Match,No Match
223,Java Software Engineers develop and maintain s...,"Java programming Java frameworks (e.g., Spring...",java,No Match,No Match,No Match,No Match
